# Ga-Ru Adjustment Workflow

This notebook demonstrates how the `gliquid` workflow can be used to adjust predictions using experimental measurements:

1. Generate an initial ML-guided prediction.
2. Overlay measured DSC points.
3. Refine liquid parameters using the measured liquidus constraints.

In [43]:
## UNCOMMENT THIS BLOCK FOR GOOGLE COLAB
# !git clone https://github.com/willwerj/gliquid_python.git
# %cd /content/gliquid_python
# !pip install -q .
# from pathlib import Path
# import gliquid.config as cfg
# cfg.set_data_dir(Path("/content/gliquid_python/data").resolve())

In [44]:
import os
from pathlib import Path
from gliquid.production_model_runner import ProductionModelRunner

# Required for fetching any uncached MP data.
os.environ.setdefault("NEW_MP_API_KEY", "YOUR_API_KEY_HERE")

# Bundle with trained production models + feature metadata.
bundle_dir = str(Path.cwd().parent / 'data' / '20260217_135723')
runner = ProductionModelRunner(bundle_dir)

## 1 - Plot predicted liquidus curve

Use latest ML model ensemble to predict non-ideal mixing parameters for the Ga-Ru system and plot the resulting phase diagram.

In [ ]:
from gliquid.binary import BinaryLiquid, BLPlotter

system = "Ga-Ru"
bl = BinaryLiquid.from_cache(system, param_format='comb-exp')

pred_params = runner.predict_system(system) + [0]
bl.update_params(pred_params)
predicted_liquidus_line = [[p[0], p[1]] for p in bl.phases[-1]['points']]  # Keep a copy for later overlay

blp = BLPlotter(bl)
blp.show('pred')   # Predicted phase diagram
blp.show('ch+g')   # DFT convex hull with liquid free-energy overlay

Ga: H_liq = 5585.0 J/mol, S_liq = 18.4384 J/(mol·K), T_fusion = 302.9 K, polymorphs = 1
Ru: H_liq = 27481.0 J/mol, S_liq = 10.5453 J/(mol·K), T_fusion = 2606.0 K, polymorphs = 1

No cached binary phase data found!
MPDS_API_KEY not found in environment variables. Proceeding without binary phase data.
From recent model; [-123232.8828125, -13.68562126159668, -33274.7998046875, 0]


No arguments specified for 't_vals', setting 't_units' to 'K'


Following this initial prediction, this system was identified as a system that our collaborators at Baylor University
were capable of conducting an assessment on using their DSC techniques.

Below is a table of measurements from their assessment of the Ru-Ga system:

## Table 1. DSC runs of the Ru-Ga system and weight % change

| Composition (at.% Ru: at.% Ga) | Solidus (±0.5 °C) | Liquidus (±0.5 °C) | Weight Change (±0.1 %) |
|-------------------------|-------------------|-------------------|------------------------|
| 5: 95                   | -                 | 1008.8            | 0.149                  |
| 15: 85a                 | 1095.5            | 1103.7            | 0.034                  |
| 15: 85b                 | 1085.9            | 1111.1            | 0.132                  |
| 15: 85c                 | -                 | 1109.6            | 0.004                  |
| 20: 80                  | 1087.6            | -                 | 0.065                  |
| 25: 75a                 | 1090.3            | -                 | 0.022                  |
| 25: 75b                 | 1090.4            | -                 | 0.086                  |
| 25: 75c                 | 1088.9            | -                 | 0.056                  |
| 30: 70                  | 1090.9            | -                 | 0.009                  |
| 35: 65a                 | -                 | -                 | 0.027                  |
| 35: 65b                 | -                 | -                 | 0.219                  |
| 35: 65c                 | -                 | -                 | 0.002                  |
| 45: 55a                 | -                 | -                 | 0.014                  |
| 45: 55b<sup>a</sup>     | -                 | -                 | 0.143                  |
| 45: 55c<sup>b</sup>     | -                 | -                 | 0.006                  |
| 50: 50                  | -                 | -                 | 0.101                  |
| 55: 45a<sup>c</sup>     | -                 | -                 | 0.028                  |
| 55: 45b<sup>d</sup>     | -                 | -                 | 0.015                  |
| 60: 40                  | -                 | -                 | 0.150                  |
| 68: 32                  | -                 | -                 | 0.008                  |
| 75: 25a<sup>a</sup>     | -                 | -                 | 0.                     |
| 75: 25b                 | -                 | -                 | 0.                     |

The letters a,b,c after the compositions are there only to distinguish samples of the same composition. The exponents are to signify samples that were prepared differently while trying to determine the preparation method that provides the most homogenous sample.<br> 
<sup>a</sup> Sample used larger Ga pieces relative to the fine Ga powder used for other samples<br>
<sup>b</sup> Sample used Ru pieces that were wet with molten Ga<br>
<sup>c, d</sup> Sample used Ru powder that was pressed into a pellet, then Ga pieces were dropped on top<br>

Credit:

Mario A. Plata,<sup>a</sup> J. Streit Smith,<sup>a</sup> and Julia Y. Chan<sup>* a</sup><br>
<sup>a</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States.<br>
<sup>*</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States;<br>
https://orcid.org/0000-0003-4434-2160; Email: Julia_Chan@baylor.edu<br>

In [46]:
# composition at% Ru, solidus temperature (C), liquidus temperature (C)
measured_points_wtpct = [[5,    None,   1008.8],
                         [15,   1095.5, 1103.7],
                         [15,   1085.9, 1111.1],
                         [15,   None,   1109.6],
                         [20,   1087.6, None],
                         [25,   1090.3, None],
                         [25,   1090.4, None],
                         [25,   1088.9, None],
                         [30,   1090.9, None],
                         [35,   None,   None],
                         [35,   None,   None],
                         [35,   None,   None],
                         [45,   None,   None],
                         [45,   None,   None],
                         [45,   None,   None],
                         [50,   None,   None],
                         [55,   None,   None],
                         [55,   None,   None],
                         [60,   None,   None],
                         [68,   None,   None],
                         [75,   None,   None],
                         [75,   None,   None]] 

## 2 - Overlay measurements on the baseline prediction

Convert repeated measurements into averaged solidus/liquidus observations at each composition, then plot the measured points against the initial predicted phase diagram.

In [47]:
def process_measured_points(measured_points_wtpct):
    """
    Average multiple measurements at the same composition.
    Return solidus points, liquidus points, and compositions with no melting.
    """
    s, l, no_melt = [], [], []
    s_dict, l_dict = {}, {}

    for wt_pct, solidus, liquidus in measured_points_wtpct:

        if solidus is None and liquidus is None and wt_pct not in no_melt:
            no_melt.append(wt_pct)
        if solidus is not None:
            if wt_pct not in s_dict:
                s.append([wt_pct, 0])
            s_dict[wt_pct] = s_dict.get(wt_pct, []) + [solidus]
        if liquidus is not None:
            if wt_pct not in l_dict:
                l.append([wt_pct, 0])
            l_dict[wt_pct] = l_dict.get(wt_pct, []) + [liquidus]

    for point in s:
        point[1] = round(sum(s_dict[point[0]]) / len(s_dict[point[0]]), 1)
    for point in l:
        point[1] = round(sum(l_dict[point[0]]) / len(l_dict[point[0]]), 1)

    return s, l, no_melt

solidus_points, liquidus_points, no_liquid_comps = process_measured_points(measured_points_wtpct)
print("Solidus points:", solidus_points)
print("Liquidus points:", liquidus_points)
print("Compositions with no melting (at. %):", no_liquid_comps)


Solidus points: [[15, 1090.7], [20, 1087.6], [25, 1089.9], [30, 1090.9]]
Liquidus points: [[5, 1008.8], [15, 1108.1]]
Compositions with no melting (at. %): [35, 45, 50, 55, 60, 68, 75]


In [48]:
import plotly.graph_objects as go

def add_experimental_traces(figure: go.Figure, inc_no_liquid=True):
    """Add measured observations to a Plotly phase-diagram figure."""
    
    if inc_no_liquid:
        # Vertical guides for compositions where no liquid phase was observed
        for i, x_val in enumerate(no_liquid_comps):
            figure.add_trace(go.Scatter(
                x=[x_val, x_val],
                y=[0, 900],
                mode='lines',
                line=dict(color='red', width=2),
                name='No Liquid Phase Observed' if i == 0 else None,
                showlegend=i == 0,
            ))
            figure.add_trace(go.Scatter(
                x=[x_val, x_val],
                y=[900, 1200],
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                showlegend=False
            ))

    figure.add_trace(go.Scatter(
            x=[point[0] for point in solidus_points][1:],
            y=[point[1] for point in solidus_points][1:],
            mode='markers',
            marker=dict(color='#ADEBB3', size=14, symbol='triangle-up', line=dict(width=1, color='black')),
            name='Solidus Observed'
    ))

    figure.add_trace(go.Scatter(
            x=[point[0] for point in liquidus_points],
            y=[point[1] for point in liquidus_points],
            mode='markers',
            marker=dict(color='cornflowerblue', size=12, symbol='square', line=dict(width=1, color='black')),
            name='Liquidus Observed'
    ))

fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.update_layout(yaxis_range=[-150, 2550])
fig.show()

## 3 - Refine liquid parameters using measured constraints

We first adjust the solid phases with the convex hull editor, then use the measured liquidus to refine the non-ideal mixing parameters. The `ConvexHullEditor` is initialized directly from the `BinaryLiquid`; its temperature slider shows the Gibbs hull `G = H - T*S`, and `apply()` writes the edited hull and phase list (with entropies) back to the object.

In [58]:
import numpy as np
from gliquid.binary import _x_vals
from gliquid.hull_editor import ConvexHullEditor

def optimize_predicted_params(bl: BinaryLiquid, known_points: list, predicted_params: list) -> tuple:

    hull_points = np.array([[p['comp'], p['enthalpy']] for p in bl.phases if 'comp' in p])
    bl.eqs['h_hull_interp'] = np.interp(_x_vals[1:-1], hull_points[:, 0], hull_points[:, 1])
    bl.comp_range_fit_lim = 0
    bl.init_error = False
    bl.max_liq_temp = max(v['T_fusion'] for v in bl.component_data.values())
    bl.min_liq_temp = min(v['T_fusion'] for v in bl.component_data.values())
    bl.digitized_liq = [[p[0], p[1]] for p in known_points]

    mae, _, _, _ = bl.calculate_deviation_metrics(ignored_ranges=False, allow_sparse_data=True)
    print(f"Known points (Kelvin): {known_points}")
    print(f"Liquidus MAE from known points: {int(mae)}K")

    print(f"Original predicted parameters: {predicted_params}")

    bl.fit_parameters(verbose=True, max_iter=128, check_phase_mismatch=False, allow_sparse_data=True, 
                      disable_inv_constrs=True, params_init=predicted_params)
    
    print(f"Adjusted parameters: {bl.get_params()}")
    return bl.get_params()


# Edit the solid phases with the convex hull editor (initialized from the BinaryLiquid).
# GaRu sits just above the T=0 DFT hull, but its configurational entropy stabilizes it at
# the assessment temperature; Ga3Ru is given a small negative entropy. Entropies are in
# eV/atom/K (to match the eV/atom energy axis); 1.057e-4 eV/atom/K = 10.2 J/mol-atom/K.
t_assess = 1108 + 273.15  # assessment temperature (K) at which the Gibbs hull is evaluated
editor = ConvexHullEditor(bl)
editor.set_temperature(t_assess)
editor.modify_entry(editor.find_entry(name='Ga3Ru'), new_entropy=-1.4, units='J/mol') # = -1.4 J/mol-atom/K
editor.modify_entry(editor.find_entry(name='GaRu'), new_entropy=8.7, units='J/mol')  # = 10.2 J/mol-atom/K
editor.display()   # interactive: inspect the Gibbs hull and tune values with the temperature slider
editor.apply()

print("Solid phase energies (eV/atom, eV/atom/K):")
for p in bl.phases:
    if 'comp' in p:
        print(f"  {p['name']:<22} Ef={p['enthalpy']/96485:+.3f}  S={p.get('entropy', 0)/96485:+.2e}")
print("-" * 30)

# Reformat measured liquidus points to [at. fraction, Kelvin]
measured_liq_points = [[p[0]/100.0, p[1]+273.15] for p in liquidus_points]
measured_liq_points.append([0.73, 1125+273.15]) # Add suspected pre-melting point to stabilize optimization
adj_params = optimize_predicted_params(bl, measured_liq_points, pred_params)
bl.temp_range = [253, 1000]
bl.update_phase_points()
blp = BLPlotter(bl)
fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.show()

Solid phase energies (eV/atom, eV/atom/K):
  alpha-Ga (orthorhombic) Ef=+0.000  S=+0.00e+00
  Ga3Ru                  Ef=-0.387  S=-1.45e-05
  Ga2Ru                  Ef=-0.413  S=+0.00e+00
  GaRu                   Ef=-0.301  S=+9.02e-05
  hcp-Ru                 Ef=+0.000  S=+0.00e+00
------------------------------
Known points (Kelvin): [[0.05, 1281.9499999999998], [0.15, 1381.25], [0.73, 1398.15]]
Liquidus MAE from known points: 34K
Original predicted parameters: [-101100, -10.91, -33100, 0]

Maximum composition range fitted: [0.05, 0.73]
Ignored composition ranges: []

Initial triangle for H-S partition constraints: [[-92893.66284002527, 42046.88179545374], [-92893.66284002527, -26820.565577003938], [-57411.440974608704, -26820.565577003938]]
--- Beginning Nelder-Mead optimization ---
Iteration # 0
Best guess: {a: -92893.66284002527, c: 42046.88179545374} f=182.7219368528896
Height of triangle = 68867.44737245768
--- 0.09200406074523926 seconds elapsed ---
Iteration # 1
Best guess: {a